<a href="https://colab.research.google.com/github/Lujain-Mahesar/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lujain-Mahesar/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [11]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
print("Connected. Testing with a small query...")
test = con.sql(f"SELECT COUNT(*) AS row_count FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')").df()
print(test)

Connected. Testing with a small query...
   row_count
0    9841378


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

My unit of analysis: one row = one report date, one client, and one content
item, from fact_content_daily_performance. I'm using month=2026-03 as my
working month (a mid-panel month, not the final sealed month reserved for
testing). Below I confirm the actual row count for that month and check the
real date range it covers.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

q1 = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(q1)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   row_count earliest_date latest_date
0    9841378    2026-03-01  2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Sorting the real fields from fact_content_daily_performance:

Features (inputs the model can use):
- gsc_impressions, gsc_avg_position, ga4_engaged_sessions, ga4_sessions,
  scroll_events

Label/proxy (what I'm predicting):
- is_low_ctr pages that get impressions but have a low click-through rate
  (gsc_clicks / gsc_impressions), among pages with enough impressions to
  trust the ratio. This is a defined rule proxy, not something directly
  observed.

Context (used for filtering/grouping, not fed to the model):
- report_date, client_hash_id, content_hash_id, month, client_has_gsc,
  gsc_data_available

Excluded (deliberately left out):
- gsc_clicks this is the exact number used to compute the ctr that defines
  my label. Including it as a feature would leak the label back into the
  inputs, the same trap I'll demonstrate on purpose in Section 4.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

q2 = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id,
           gsc_impressions, gsc_clicks, gsc_avg_position,
           ga4_engaged_sessions, ga4_sessions, scroll_events,
           client_has_gsc, gsc_data_available
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    LIMIT 5
""").df()
q2

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_engaged_sessions,ga4_sessions,scroll_events,client_has_gsc,gsc_data_available
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>,<NA>,True,True
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>,<NA>,True,True
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>,<NA>,True,True
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>,<NA>,True,True
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>,<NA>,True,True


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verifying my contract claims with real queries on month=2026-03:

1. Grain check confirming one row really is one (report_date, client,
   content item) combination, with no duplicates.
2. Row count and date span already 9,841,378 rows from 2026-03-01 to
   2026-03-31 (shown in Section 1), reconfirmed here in context.
3. Availability check filtering with gsc_data_available IS TRUE, to see
   how many rows actually have usable GSC data versus rows where it's
   missing/false.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 1. Grain check: one row = one (report_date, client_hash_id, content_hash_id)?
grain_check = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_keys
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print("Grain check (total_rows should equal distinct_keys):")
print(grain_check)

# 2. Row count + date span (reconfirmed)
counts = con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS earliest, MAX(report_date) AS latest
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print("\nRow count + date span:")
print(counts)

# 3. Availability check with IS TRUE
availability = con.sql(f"""
    SELECT COUNT(*) AS rows_with_gsc_data
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
""").df()
print("\nRows with gsc_data_available IS TRUE:")
print(availability)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain check (total_rows should equal distinct_keys):
   total_rows  distinct_keys
0     9841378        9841378

Row count + date span:
   row_count   earliest     latest
0    9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Rows with gsc_data_available IS TRUE:
   rows_with_gsc_data
0             3611061


In [15]:
# The leakage trap: build a quick label, then a quick score using a legit
# feature set, then deliberately add a leaky column and watch the score
# jump unrealistically.

import pandas as pd
from sklearn.tree import DecisionTreeClassifier

sample = con.sql(f"""
    SELECT gsc_impressions, gsc_clicks, gsc_avg_position,
           ga4_engaged_sessions, ga4_sessions, scroll_events
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
    LIMIT 50000
""").df()

# Define the label: low CTR (bottom 25% of clicks/impressions)
sample["ctr"] = sample["gsc_clicks"] / sample["gsc_impressions"]
threshold = sample["ctr"].quantile(0.25)
sample["is_low_ctr"] = (sample["ctr"] <= threshold).astype(int)

# HONEST version: features that don't include clicks or ctr
honest_features = ["gsc_impressions", "gsc_avg_position", "ga4_engaged_sessions", "ga4_sessions", "scroll_events"]
X_honest = sample[honest_features].fillna(0)
y = sample["is_low_ctr"]

tree_honest = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_honest.fit(X_honest, y)
honest_score = tree_honest.score(X_honest, y)
print(f"Honest accuracy (no leakage): {honest_score:.3f}")

# LEAKY version: deliberately add gsc_clicks, which is literally used to compute the label
X_leaky = sample[honest_features + ["gsc_clicks"]].fillna(0)
tree_leaky = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_leaky.fit(X_leaky, y)
leaky_score = tree_leaky.score(X_leaky, y)
print(f"Leaky accuracy (gsc_clicks included): {leaky_score:.3f}  <- looks amazing, but it's fake")

print(f"\nAdding gsc_clicks jumped accuracy from {honest_score:.3f} to {leaky_score:.3f}.")
print("That's because gsc_clicks is literally the numerator used to compute ctr,")
print("which is what the label is based on. This is leakage — I'm keeping only")
print("the honest feature set going forward.")

Honest accuracy (no leakage): 0.889
Leaky accuracy (gsc_clicks included): 1.000  <- looks amazing, but it's fake

Adding gsc_clicks jumped accuracy from 0.889 to 1.000.
That's because gsc_clicks is literally the numerator used to compute ctr,
which is what the label is based on. This is leakage — I'm keeping only
the honest feature set going forward.


The leakage trap: I deliberately added gsc_clicks as a feature, even though
I already flagged it as excluded in Section 2. As expected, accuracy jumped
sharply once it was included, because gsc_clicks is literally the number
used to compute the ctr that defines my label, the model isn't learning a
real pattern, it's just reading the answer key. I'm removing it and keeping
only the honest feature set (gsc_impressions, gsc_avg_position,
ga4_engaged_sessions, ga4_sessions, scroll_events) going forward.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

What this data can never tell you:

1. Availability gap only about 36.7% of rows (3,611,061 out of 9,841,378)
   have gsc_data_available IS TRUE. That means for most rows in a given
   month, I don't actually have reliable GSC signal, so any model trained
   naively on the full table would be learning largely from missing/zero
   values unless I filter for availability first.

2. Unbalanced client history per the dataset card, different clients have
   different amounts of history (see dim_clients.gsc_data_start /
   ga4_data_start). A newer client might only have a few months of data,
   while an older one has years. Comparing "declining" trends across clients
   without accounting for this could unfairly flag newer clients as
   "declining" just because they have less history to establish a baseline.

3. Sample month is not representative the _sample file is the final month
   (June 2026), and per the assignment's own warning, that's the natural
   outcome window for any past-to-future label. I deliberately used
   month=2026-03 instead, and I'm treating the final month as a sealed test
   set I shouldn't touch for developing label logic.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

pct_available = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS available_rows,
        ROUND(100.0 * SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_available
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(pct_available)

   total_rows  available_rows  pct_available
0     9841378       3611061.0           36.7


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.